# MS-GAT v5: how much do duplicates and single-source testing inflate brain tumor MRI accuracy?

This notebook reruns the whole study in one go so it can run overnight on Kaggle ("Save Version" > "Save & Run All").
Every number it prints is saved into `/kaggle/working/msgat_v5_results/` and zipped into
`/kaggle/working/msgat_v5_results.zip`. Download that zip in the morning and send it to Claude.

**What it runs**

| Part | What | Why |
|---|---|---|
| Data | Finds the 3 sources, exact-hash dedup, flip-aware near-copy groups (Hamming <= 4 of 256 bits) | Same data as the paper, plus the groups needed for a clean split |
| A | Main run on the **random split**, a **near-copy-free group split**, and (if Figshare is attached) a **patient + near-copy split** | RQ1 as a controlled experiment, not just an association |
| B | Ablation on the random split (k, graph mode, class weights, single-scale, CLAHE off, GCN instead of GAT) | Same table as the paper, plus a GCN control |
| Patients | Optional: matches images to the Figshare dataset (Cheng et al.) to recover patient IDs | Adds a patient-level split when the Figshare `.mat` files are attached |
| C | 5 seeds x {random, group, patient split} x {ResNet50, MLP, GAT-T, GAT-I, k-NN} | Puts the MLP control into the seed experiment |
| D | Leave-one-source-out, 5 seeds, per-image predictions saved | Rescores each held-out source without its cross-source near-copies |
| E | Size- and class-matched control for the MRI.zip fold | Separates "small, lopsided training set" from "new source" |

**Settings before running:** Accelerator = GPU T4 x2 (or one T4), Internet = On, the dataset with the three zips attached.
Expected runtime on a T4: about 3.5 to 4.5 hours. A safety timer stops new work after 11 hours and still saves everything.

In [ ]:
# ===== 0. Install the few packages Kaggle does not have =====
import os, sys, subprocess, importlib

def ensure(pip_name, module):
    try:
        importlib.import_module(module)
    except ImportError:
        print("installing", pip_name)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

ensure("torch_geometric", "torch_geometric")
ensure("imagehash", "imagehash")
ensure("grad-cam", "pytorch_grad_cam")      # optional: only used for the Grad-CAM figure

In [ ]:
# ===== 1. Config =====
import time, json, copy, re, zipfile, math, random, traceback, warnings, hashlib
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

INPUT_ROOT = os.environ.get("MSGAT_INPUT", "/kaggle/input")
WORK = os.environ.get("MSGAT_WORK", "/kaggle/working")
SMOKE = os.environ.get("MSGAT_SMOKE", "0") == "1"      # tiny budgets, used only to test the notebook

CFG = dict(
    seeds=[42, 7, 123, 2024, 99],
    main_seed=42,
    bb_main=(12, 4),        # backbone (max epochs, patience) for Part A, same as the paper
    bb_short=(8, 3),        # backbone budget for Parts B (CLAHE off), C, D, E
    gnn_main=(100, 20),     # GAT / MLP budget for Part A
    gnn_short=(60, 12),     # GAT / MLP budget for Parts B, C, D, E
    k=8,                    # neighbor search size (includes the node itself in the transductive graph, as in the paper)
    near_T=4,               # Hamming threshold (bits of 256) for "near-copy", flips included
    audit_T=[0, 4, 8, 16, 24, 32],
    batch=32,
    workers=min(4, os.cpu_count() or 2),
    deadline_hours=11.0,    # stop starting new work after this, then save
    run=dict(A=True, B=True, C=True, D=True, E=True),
)
if SMOKE:
    CFG.update(seeds=[42, 7], bb_main=(1, 1), bb_short=(1, 1), gnn_main=(3, 2), gnn_short=(3, 2), workers=0)

OUT = os.path.join(WORK, "msgat_v5_results")
THUMBS = os.path.join(OUT, "thumbs")
DATA_DIR = os.path.join(WORK, "data_extracted")
for d in (OUT, THUMBS, DATA_DIR):
    os.makedirs(d, exist_ok=True)

T_START = time.time()
RESULTS = {"config": CFG, "smoke": SMOKE, "errors": {}, "timings_min": {}}


def log(*a):
    msg = f"[{(time.time() - T_START) / 60:7.1f} min] " + " ".join(str(x) for x in a)
    print(msg, flush=True)
    with open(os.path.join(OUT, "run_log.txt"), "a") as f:
        f.write(msg + "\n")


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def save_results():
    with open(os.path.join(OUT, "results.json"), "w") as f:
        json.dump(RESULTS, f, indent=1, default=_json_default)


def time_left_ok(margin_min=0):
    return (time.time() - T_START) / 3600 < CFG["deadline_hours"] - margin_min / 60


def run_stage(name, fn):
    """Runs one part. If it crashes, the error is saved and the notebook moves on to the next part."""
    if not time_left_ok():
        log(f"SKIPPED {name}: past the {CFG['deadline_hours']} h safety deadline")
        RESULTS["errors"][name] = "skipped: deadline"
        save_results()
        return None
    t0 = time.time()
    log(f"===== START {name} =====")
    try:
        out = fn()
        log(f"===== DONE {name} ({(time.time() - t0) / 60:.1f} min) =====")
        return out
    except Exception as e:
        RESULTS["errors"][name] = f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        log(f"!!!!! FAILED {name}: {type(e).__name__}: {e}")
        print(traceback.format_exc())
        return None
    finally:
        RESULTS["timings_min"][name] = round((time.time() - t0) / 60, 2)
        save_results()

log("SMOKE test mode" if SMOKE else "Full run", "| input:", INPUT_ROOT, "| output:", OUT)

In [ ]:
# ===== 2. Imports, device, seeds =====
from PIL import Image, ImageOps
import cv2
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_recall_fscore_support,
                             f1_score, cohen_kappa_score, roc_auc_score)
from scipy.stats import binomtest, ttest_rel
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import imagehash
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.utils import dropout_edge

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_OK = device.type == "cuda"
PIN = device.type == "cuda"
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp")
CLASSES = ["glioma", "notumor", "meningioma", "pituitary"]


def _norm(s):
    return re.sub(r"[^a-z]", "", str(s).lower())


CLASS_ALIASES = {_norm(k): v for k, v in {"glioma": 0, "notumor": 1, "notumour": 1, "normal": 1, "healthy": 1,
                                          "nontumor": 1, "meningioma": 2, "pituitary": 3}.items()}


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


import torch_geometric, sklearn
RESULTS["versions"] = {"torch": torch.__version__, "torch_geometric": torch_geometric.__version__,
                       "sklearn": sklearn.__version__, "python": sys.version.split()[0],
                       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
log("device:", RESULTS["versions"]["gpu"])
set_seed(CFG["main_seed"])

## Data: find the three sources, check the counts, deduplicate
Upload the three original zips (`MRI.zip`, `archive.zip`, and `PMRAM Bangladeshi Brain Cancer - MRI Dataset.zip`) as one
Kaggle dataset. Kaggle may unzip them for you; this cell handles both cases, opens the nested `Raw Data.zip`
inside PMRAM, and never touches `Augmented Data.zip`. If a count is wrong it stops right away and prints the folder tree.

In [ ]:
# ===== 3. Find and load the data =====
def print_tree(root, depth=3, max_entries=12):
    root = os.path.abspath(root)
    for dp, dns, fns in os.walk(root):
        lvl = dp[len(root):].count(os.sep)
        if lvl >= depth:
            dns[:] = []
            continue
        dns.sort()
        print("  " * lvl + os.path.basename(dp) + f"/  ({len(fns)} files)")
        for d in dns[:max_entries]:
            pass


def extract_zips(roots):
    seen = set()
    for _ in range(4):                       # handles zips inside zips
        new = False
        for root in roots:
            if not os.path.isdir(root):
                continue
            for dp, dns, fns in os.walk(root):
                dns.sort()
                for f in sorted(fns):
                    if not f.lower().endswith(".zip"):
                        continue
                    zp = os.path.join(dp, f)
                    if zp in seen:
                        continue
                    seen.add(zp)
                    if "augment" in _norm(f):
                        log("not extracting (augmented copies):", zp)
                        continue
                    tag = hashlib.md5(zp.encode()).hexdigest()[:6]
                    dest = os.path.join(DATA_DIR, (_norm(os.path.splitext(f)[0]) or "zip") + "_" + tag)
                    if not os.path.exists(dest + ".done"):
                        log("extracting", zp, "->", dest)
                        with zipfile.ZipFile(zp) as z:
                            z.extractall(dest)
                        open(dest + ".done", "w").close()
                        new = True
        if not new:
            break


def collect(root, exclude_tokens=("augment", "segmentation", "mask")):
    paths, labs = [], []
    for dp, dns, fns in os.walk(root):
        dns.sort()
        if any(t in _norm(dp) for t in exclude_tokens):
            continue
        key = _norm(os.path.basename(dp))
        if key in CLASS_ALIASES:
            for f in sorted(fns):
                if f.lower().endswith(VALID_EXTS):
                    paths.append(os.path.join(dp, f)); labs.append(CLASS_ALIASES[key])
    return paths, labs


def find_dirs(roots, pred):
    out = []
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dp, dns, fns in os.walk(r):
            dns.sort()
            if pred(dp, dns):
                out.append(dp)
    return out


def load_sources():
    extract_zips([INPUT_ROOT, DATA_DIR])
    roots = [INPUT_ROOT, DATA_DIR]
    bad = ("brisc", "pmram", "bangladesh", "rawdata", "augment")

    # --- MRI.zip: a "Training" folder with a "Testing" sibling ---
    def is_mri(dp, dns):
        if _norm(os.path.basename(dp)) != "training" or any(b in _norm(dp) for b in bad):
            return False
        parent = os.path.dirname(dp)
        return any(_norm(x) == "testing" and os.path.isdir(os.path.join(parent, x)) for x in os.listdir(parent))
    cands = find_dirs(roots, is_mri)
    mri = None
    for c in cands:
        parent = os.path.dirname(c)
        test_dir = [os.path.join(parent, x) for x in sorted(os.listdir(parent)) if _norm(x) == "testing"][0]
        tr, te = collect(c), collect(test_dir)
        if SMOKE or (len(tr[0]) == 5600 and len(te[0]) == 1600):
            mri = (tr, te, c, test_dir); break
        log("MRI.zip candidate rejected (counts):", c, len(tr[0]), len(te[0]))

    # --- BRISC 2025 ---
    cands = find_dirs(roots, lambda dp, dns: "brisc" in _norm(os.path.basename(dp)))
    cands = sorted(cands, key=lambda p: p.count(os.sep))
    cands += find_dirs(roots, lambda dp, dns: _norm(os.path.basename(dp)) in ("classificationtask",)
                       or _norm(os.path.basename(dp)).startswith("archive"))
    brisc = None
    for c in cands:
        p = collect(c)
        if SMOKE or len(p[0]) == 6000:
            brisc = (p, c); break
        log("BRISC candidate rejected (count):", c, len(p[0]))

    # --- PMRAM Raw Data only ---
    cands = find_dirs(roots, lambda dp, dns: _norm(os.path.basename(dp)).startswith("rawdata") and "augment" not in _norm(dp))
    pmram = None
    for c in cands:
        p = collect(c)
        if (SMOKE and len(p[0]) > 0) or 1400 <= len(p[0]) <= 1700:
            pmram = (p, c); break
        log("PMRAM candidate rejected (count):", c, len(p[0]))

    missing = [n for n, v in (("MRI.zip", mri), ("BRISC (archive.zip)", brisc), ("PMRAM Raw Data", pmram)) if v is None]
    if missing:
        print("\nFolder tree of the input:")
        print_tree(INPUT_ROOT, depth=4)
        print("\nFolder tree of extracted zips:")
        print_tree(DATA_DIR, depth=4)
        raise RuntimeError(f"Could not find: {missing}. Check that the three zips are attached as a Kaggle dataset "
                           f"(Add Input). Expected 7,200 MRI.zip images (5,600 Training + 1,600 Testing), 6,000 BRISC "
                           f"classification images, and about 1,505 PMRAM Raw Data images.")

    raw_paths, raw_labels, raw_source = [], [], []
    for name, (p, l) in [("ds1_primary_train", mri[0]), ("ds1_primary_test", mri[1]),
                         ("ds2_brisc2025", brisc[0]), ("ds3_pmram_raw", pmram[0])]:
        raw_paths += p; raw_labels += l; raw_source += [name] * len(p)
        log(f"{name}: {len(p)} images", {c: int(np.sum(np.array(l) == i)) for i, c in enumerate(CLASSES)})
    RESULTS["data_roots"] = {"mri_train": mri[2], "mri_test": mri[3], "brisc": brisc[1], "pmram": pmram[1]}
    return raw_paths, np.array(raw_labels), np.array(raw_source)


raw_paths, raw_labels, raw_source = load_sources()
log("TOTAL raw:", len(raw_paths))
assert SMOKE or (raw_source == "ds3_pmram_raw").sum() < 2000, "PMRAM augmented copies got in. Stop and check the dataset."

In [ ]:
# ===== 4. Exact-hash dedup (same rule as the paper) + flipped hashes for the near-copy audit =====
def phash_hex(img):
    return str(imagehash.phash(img, hash_size=16))


def hex_to_bits(hexes):
    return np.unpackbits(np.frombuffer(b"".join(bytes.fromhex(h) for h in hexes), dtype=np.uint8).reshape(len(hexes), -1), axis=1)


t0 = time.time()
raw_hash = []
for p in raw_paths:
    try:
        raw_hash.append(phash_hex(Image.open(p).convert("L")))
    except Exception:
        raw_hash.append(None)
log(f"hashed {len(raw_paths)} images in {time.time() - t0:.0f}s; unreadable: {sum(h is None for h in raw_hash)}")

first_of = {}
for i, h in enumerate(raw_hash):
    if h is not None and h not in first_of:
        first_of[h] = i
keep = np.array(sorted(first_of.values()))
image_paths = np.array([raw_paths[i] for i in keep])
labels = raw_labels[keep]
src_full = raw_source[keep]
image_source = np.array([s.split("_")[0] for s in src_full])          # ds1 / ds2 / ds3
N = len(image_paths)
log(f"unique after exact-hash dedup: {N} ({100 * (1 - N / len(raw_paths)):.1f}% removed)")

t0 = time.time()
hex_o = [raw_hash[i] for i in keep]
hex_h, hex_v = [], []
for p in image_paths:
    im = Image.open(p).convert("L")
    hex_h.append(phash_hex(ImageOps.mirror(im))); hex_v.append(phash_hex(ImageOps.flip(im)))
B_O, B_H, B_V = hex_to_bits(hex_o), hex_to_bits(hex_h), hex_to_bits(hex_v)
log(f"flipped hashes in {time.time() - t0:.0f}s; hash bits per image: {B_O.shape[1]}")

RESULTS["data"] = {
    "raw_total": len(raw_paths),
    "raw_per_source_class": pd.crosstab(raw_source, raw_labels).rename(columns=dict(enumerate(CLASSES))).to_dict(orient="index"),
    "unique_total": N,
    "unique_per_class": {c: int((labels == i).sum()) for i, c in enumerate(CLASSES)},
    "unique_per_source_class": pd.crosstab(src_full, labels).rename(columns=dict(enumerate(CLASSES))).to_dict(orient="index"),
}
save_results()

In [ ]:
# ===== 5. Near-copy distances and flip-aware groups =====
_BO_t = torch.tensor(B_O, dtype=torch.float32, device=device)
_BH_t = torch.tensor(B_H, dtype=torch.float32, device=device)
_BV_t = torch.tensor(B_V, dtype=torch.float32, device=device)


def nearest_copy(query_idx, ref_idx, chunk=2048, flips=True):
    """For each query image: smallest Hamming distance to any ref image (query also tried flipped),
    and which ref image it is. Returns (dist, nearest_global_idx, dist_without_flips)."""
    query_idx, ref_idx = np.asarray(query_idx), np.asarray(ref_idx)
    R = _BO_t[ref_idx]; rs = R.sum(1)
    best = np.empty(len(query_idx), np.int32); arg = np.empty(len(query_idx), np.int64); best_o = np.empty(len(query_idx), np.int32)
    for s in range(0, len(query_idx), chunk):
        q = query_idx[s:s + chunk]
        dmin, amin = None, None
        for vi, Q in enumerate((_BO_t, _BH_t, _BV_t) if flips else (_BO_t,)):
            Qq = Q[q]
            D = Qq.sum(1, keepdim=True) + rs[None, :] - 2.0 * (Qq @ R.T)
            same = torch.tensor(q, device=device)[:, None] == torch.tensor(ref_idx, device=device)[None, :]
            D = D.masked_fill(same, 1e9)                       # never match an image to itself
            d, a = D.min(1)
            if vi == 0:
                best_o[s:s + chunk] = d.round().int().cpu().numpy()
            if dmin is None:
                dmin, amin = d, a
            else:
                better = d < dmin; dmin = torch.where(better, d, dmin); amin = torch.where(better, a, amin)
        best[s:s + chunk] = dmin.round().int().cpu().numpy(); arg[s:s + chunk] = ref_idx[amin.cpu().numpy()]
    return best, arg, best_o


def near_copy_pairs(T, chunk=1024):
    rows, cols = [], []
    allidx = np.arange(N)
    rs = _BO_t.sum(1)
    for s in range(0, N, chunk):
        q = allidx[s:s + chunk]
        Dm = None
        for Q in (_BO_t, _BH_t, _BV_t):
            Qq = Q[q]
            D = Qq.sum(1, keepdim=True) + rs[None, :] - 2.0 * (Qq @ _BO_t.T)
            Dm = D if Dm is None else torch.minimum(Dm, D)
        Dm[torch.arange(len(q), device=device), torch.tensor(q, device=device)] = 1e9
        r, c = torch.nonzero(Dm <= T + 0.5, as_tuple=True)
        rows.append(r.cpu().numpy() + s); cols.append(c.cpu().numpy())
    return np.concatenate(rows), np.concatenate(cols)


def stage_groups():
    global groups
    r, c = near_copy_pairs(CFG["near_T"])
    A = coo_matrix((np.ones(len(r)), (r, c)), shape=(N, N))
    n_groups, groups = connected_components(A, directed=False)
    sizes = np.bincount(groups)
    mixed = sum(len(set(labels[groups == g])) > 1 for g in np.where(sizes > 1)[0])
    info = {"threshold_bits": CFG["near_T"], "pairs": int(len(r)), "n_groups": int(n_groups),
            "images_in_multi_groups": int((sizes[groups] > 1).sum()),
            "largest_group": int(sizes.max()), "size_hist": {int(k): int(v) for k, v in zip(*np.unique(sizes, return_counts=True))},
            "mixed_label_groups": int(mixed)}
    RESULTS["groups"] = info
    log("near-copy groups:", {k: v for k, v in info.items() if k != "size_hist"})
    if sizes.max() > 0.05 * N:
        log("WARNING: one near-copy group holds more than 5% of the images; the group split will be uneven.")
    pd.DataFrame({"idx": np.arange(N), "path": image_paths, "source": src_full, "label": labels, "group": groups,
                  "phash": hex_o}).to_csv(os.path.join(OUT, "image_index.csv"), index=False)


groups = np.arange(N)
run_stage("groups", stage_groups)

## Patient IDs from the original Figshare dataset (optional)
The glioma, meningioma and pituitary images in MRI.zip (and BRISC, which was built from it) come from the
Figshare dataset of Cheng et al. (3,064 slices, 233 patients), which keeps patient IDs. If those `.mat` files are
attached, each image is matched to a Figshare slice (normalized cross-correlation >= 0.95 on 64x64 thumbnails,
all 8 rotations/flips), and a third split keeps every patient and every near-copy group on one side.
If they are not attached, this part is skipped and everything else still runs.

In [ ]:
PATIENT_OK = False
pgroups = None
pid_of = np.array([""] * N, dtype=object)


def _thumb_vec(a):
    a = cv2.resize(np.asarray(a, dtype=np.float32), (64, 64), interpolation=cv2.INTER_AREA).reshape(-1)
    a = a - a.mean()
    return a / (np.linalg.norm(a) + 1e-8)


def stage_patients():
    global PATIENT_OK, pgroups, pid_of
    mats = {}
    for r in (INPUT_ROOT, DATA_DIR):
        for dp, dns, fns in os.walk(r):
            dns.sort()
            for f in sorted(fns):
                if re.fullmatch(r"\d+\.mat", f):
                    mats.setdefault(int(f[:-4]), os.path.join(dp, f))
    if len(mats) < (10 if SMOKE else 1000):
        log(f"Figshare .mat files not found ({len(mats)}); skipping the patient-level split.")
        RESULTS["patients"] = {"available": False, "n_mat_files": len(mats)}
        return
    import h5py
    fs_lab_map = {1: 2, 2: 0, 3: 3}                   # Figshare: 1 meningioma, 2 glioma, 3 pituitary
    T, t_pid, t_lab, t_num = [], [], [], []
    for num, path in sorted(mats.items()):
        with h5py.File(path, "r") as f:
            cj = f["cjdata"]
            img = np.array(cj["image"], dtype=np.float32)
            pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).reshape(-1) if int(c) > 0).strip()
            lab = fs_lab_map[int(np.array(cj["label"]).reshape(-1)[0])]
        for k in range(4):                            # all 8 rotations / reflections
            for flip in (False, True):
                v = np.rot90(img.T if flip else img, k)
                T.append(_thumb_vec(v)); t_pid.append(pid); t_lab.append(lab); t_num.append(num)
    T = torch.tensor(np.stack(T), device=device)
    log(f"loaded {len(mats)} Figshare slices ({len(set(t_pid))} patients)")
    best = np.zeros(N, np.float32); arg = np.zeros(N, np.int64)
    for s0 in range(0, N, 512):
        Q = torch.tensor(np.stack([_thumb_vec(np.array(Image.open(p).convert("L"))) for p in image_paths[s0:s0 + 512]]), device=device)
        v, a = (Q @ T.T).max(1)
        best[s0:s0 + 512] = v.cpu().numpy(); arg[s0:s0 + 512] = a.cpu().numpy()
    matched = best >= 0.95
    t_pid, t_lab, t_num = np.array(t_pid, dtype=object), np.array(t_lab), np.array(t_num)
    pid_of = np.where(matched, t_pid[arg], "").astype(object)
    label_agree = float((t_lab[arg][matched] == labels[matched]).mean()) if matched.any() else None
    # groups = near-copy groups joined with "same patient"
    r, c = near_copy_pairs(CFG["near_T"])
    rows, cols = list(r), list(c)
    for pid in set(pid_of[matched]):
        ids = np.where(pid_of == pid)[0]
        rows += list(ids[:-1]); cols += list(ids[1:])
    A = coo_matrix((np.ones(len(rows)), (np.array(rows), np.array(cols))), shape=(N, N))
    _, pgroups = connected_components(A, directed=False)
    sizes = np.bincount(pgroups)
    RESULTS["patients"] = {
        "available": True, "n_figshare_slices": len(mats), "n_patients_figshare": int(len(set(t_pid))),
        "ncc_threshold": 0.95, "n_images_matched": int(matched.sum()),
        "matched_per_source_class": pd.crosstab(image_source[matched], labels[matched]).rename(columns=dict(enumerate(CLASSES))).to_dict(orient="index"),
        "n_patients_matched": int(len(set(pid_of[matched]))), "label_agreement_with_figshare": label_agree,
        "n_distinct_figshare_slices_matched": int(len(set(t_num[arg][matched]))),
        "largest_patient_group": int(sizes.max()), "n_groups": int(len(sizes)),
        "best_ncc_share_at_least": {str(t): float((best >= t).mean()) for t in (0.80, 0.90, 0.95, 0.98)}}
    pd.DataFrame({"idx": np.arange(N), "figshare_ncc": best, "figshare_slice": np.where(matched, t_num[arg], -1),
                  "pid": pid_of}).to_csv(os.path.join(OUT, "figshare_matches.csv"), index=False)
    PATIENT_OK = True
    log("patient matching:", {k: v for k, v in RESULTS["patients"].items() if k != "matched_per_source_class"})


run_stage("patients", stage_patients)

In [ ]:
# ===== 6. Splits =====
def random_split(seed):
    idx = np.arange(N)
    trv, te = train_test_split(idx, test_size=0.20, random_state=seed, stratify=labels)
    tr, va = train_test_split(trv, test_size=0.20, random_state=seed, stratify=labels[trv])
    return np.sort(tr), np.sort(va), np.sort(te)


def group_split(seed, pool=None, grp=None):
    grp = groups if grp is None else grp
    pool = np.arange(N) if pool is None else np.asarray(pool)
    sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv_l, te_l = next(sg.split(np.zeros(len(pool)), labels[pool], grp[pool]))
    trv, te = pool[trv_l], pool[te_l]
    sg2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_l, va_l = next(sg2.split(np.zeros(len(trv)), labels[trv], grp[trv]))
    tr, va = trv[tr_l], trv[va_l]
    assert not (set(grp[te]) & set(grp[trv])), "a group crosses the test boundary"
    assert not (set(grp[tr]) & set(grp[va])), "a group crosses the train/val boundary"
    return np.sort(tr), np.sort(va), np.sort(te)


def make_split(kind, seed):
    if kind == "random":
        return random_split(seed)
    if kind == "group":
        return group_split(seed)
    if kind == "patient":
        return group_split(seed, grp=pgroups)
    raise ValueError(kind)


def split_kinds():
    return ["random", "group"] + (["patient"] if PATIENT_OK else [])


def split_info(tr, va, te):
    d, _, d_o = nearest_copy(te, np.concatenate([tr, va]))
    return {"sizes": {"train": len(tr), "val": len(va), "test": len(te)},
            "test_per_class": {c: int((labels[te] == i).sum()) for i, c in enumerate(CLASSES)},
            "test_per_source": {s: int((image_source[te] == s).sum()) for s in sorted(set(image_source))},
            "test_near_copy_share": {int(T): float((d <= T).mean()) for T in CFG["audit_T"]},
            "test_near_copy_share_no_flips": {int(T): float((d_o <= T).mean()) for T in CFG["audit_T"]},
            **({"patient_leak": patient_leak(tr, va, te)} if PATIENT_OK else {})}, d, d_o


def patient_leak(tr, va, te):
    seen = set(pid_of[np.concatenate([tr, va])]) - {""}
    t = [p for p in pid_of[te] if p]
    return {"test_images_with_patient_id": len(t), "share_whose_patient_is_in_training": float(np.mean([p in seen for p in t])) if t else None}

## Models: backbone, multi-scale features, graph, GAT, MLP, GCN, k-NN vote
Same architecture and training recipe as the paper. Every predictor returns a full-length array indexed by image,
so two models are always compared on the same images in the same order.

In [ ]:
# ===== 7. Backbone =====
def enhance_mri(image):
    g = np.array(image.convert("L"))
    g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
    return Image.fromarray(cv2.cvtColor(g, cv2.COLOR_GRAY2RGB))


NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
                               transforms.ColorJitter(brightness=0.1, contrast=0.1), transforms.ToTensor(), NORM])
eval_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), NORM])


class MRIDataset(Dataset):
    def __init__(self, idx, tf, use_clahe=True):
        self.idx, self.tf, self.use_clahe = np.asarray(idx), tf, use_clahe
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        im = Image.open(image_paths[j]).convert("RGB")
        if self.use_clahe:
            im = enhance_mri(im)
        return self.tf(im), int(labels[j])


def loader(idx, tf, shuffle, bs, use_clahe=True):
    return DataLoader(MRIDataset(idx, tf, use_clahe), batch_size=bs, shuffle=shuffle, num_workers=CFG["workers"],
                      pin_memory=PIN, persistent_workers=False)


class NonFinite(RuntimeError):
    pass


def new_resnet():
    m = models.resnet50(weights=None if SMOKE else models.ResNet50_Weights.DEFAULT)
    for name, p in m.named_parameters():
        p.requires_grad = "layer4" in name or "fc" in name
    m.fc = torch.nn.Linear(m.fc.in_features, len(CLASSES))
    return m.to(device)


def _train_backbone(tr, va, seed, epochs, patience, use_clahe, amp):
    set_seed(seed)
    m = new_resnet()
    tl, vl = loader(tr, train_tf, True, CFG["batch"], use_clahe), loader(va, eval_tf, False, 64, use_clahe)
    crit = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(epochs):
        m.train(); run = 0.0
        for x, y in tl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.autocast(device_type=device.type, enabled=amp):
                loss = crit(m(x), y)
            if not torch.isfinite(loss):
                raise NonFinite("non-finite training loss")
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        m.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in vl:
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type=device.type, enabled=amp):
                    correct += (m(x).argmax(1) == y).sum().item()
                total += len(y)
        acc = correct / total
        hist.append({"epoch": ep + 1, "train_loss": run / max(1, len(tl)), "val_acc": acc})
        if acc > best:
            best, best_state, bad = acc, copy.deepcopy(m.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    m.load_state_dict(best_state)
    return m, best, hist


def train_backbone(tr, va, seed, budget, use_clahe=True, amp=None):
    amp = AMP_OK if amp is None else (amp and AMP_OK)
    epochs, patience = budget
    try:
        m, best, hist = _train_backbone(tr, va, seed, epochs, patience, use_clahe, amp)
    except NonFinite:
        if not amp:
            raise
        log("non-finite loss in fp16; retrying this backbone in fp32")
        m, best, hist = _train_backbone(tr, va, seed, epochs, patience, use_clahe, False)
        amp = False
    return {"model": m, "best_val_acc": best, "history": hist, "amp": amp, "use_clahe": use_clahe}


@torch.no_grad()
def predict_backbone(bb, idx):
    m = bb["model"]; m.eval()
    probs = []
    for x, _ in loader(idx, eval_tf, False, 64, bb["use_clahe"]):
        with torch.autocast(device_type=device.type, enabled=bb["amp"]):
            probs.append(F.softmax(m(x.to(device)).float(), 1).cpu().numpy())
    P = np.concatenate(probs)
    full_pred, full_prob = np.full(N, -1), np.full((N, 4), np.nan)
    full_pred[idx] = P.argmax(1); full_prob[idx] = P
    return full_pred, full_prob


@torch.no_grad()
def extract_features(bb, multiscale=True):
    m = bb["model"]; m.eval(); acts = {}
    hooks = [getattr(m, n).register_forward_hook(lambda mod, i, o, n=n: acts.__setitem__(n, o)) for n in ("layer2", "layer3", "layer4")]
    feats = []
    try:
        for x, _ in loader(np.arange(N), eval_tf, False, 64, bb["use_clahe"]):
            with torch.autocast(device_type=device.type, enabled=bb["amp"]):
                m(x.to(device))
            names = ("layer2", "layer3", "layer4") if multiscale else ("layer4",)
            feats.append(torch.cat([acts[n].float().mean((2, 3)) for n in names], 1).cpu().numpy())
    finally:
        for h in hooks:
            h.remove()
    Fm = np.concatenate(feats).astype(np.float32)
    if not np.isfinite(Fm).all():
        if bb["amp"]:
            log("non-finite features in fp16; re-extracting in fp32")
            return extract_features({**bb, "amp": False}, multiscale)
        raise NonFinite("non-finite features in fp32")
    return Fm

In [ ]:
# ===== 8. Graph, GAT, GCN, MLP, k-NN =====
def build_graph(feats, tr, va, te, k=8, mode="transductive"):
    """Same rule as the paper: edges point neighbor -> query (PyG sends messages edge_index[0] -> edge_index[1]).
    Transductive: neighbor search over all images, k includes the node itself (so 7 real neighbors + self-loop).
    Inductive: neighbors only among train+val; test images receive edges from k train+val images and send none."""
    n = len(feats)
    if mode == "transductive":
        _, ind = NearestNeighbors(n_neighbors=k, metric="cosine").fit(feats).kneighbors(feats)
        src, dst = ind.reshape(-1), np.repeat(np.arange(n), k)
    elif mode == "inductive":
        allowed = np.concatenate([tr, va])
        nn_ = NearestNeighbors(n_neighbors=k, metric="cosine").fit(feats[allowed])
        _, ia = nn_.kneighbors(feats[allowed]); _, it = nn_.kneighbors(feats[te])
        src = np.concatenate([allowed[ia.reshape(-1)], allowed[it.reshape(-1)]])
        dst = np.concatenate([np.repeat(allowed, k), np.repeat(te, k)])
        assert not np.isin(src, te).any(), "inductive graph leaks test nodes"
    else:
        raise ValueError(mode)
    g = Data(x=torch.tensor(feats, dtype=torch.float), edge_index=torch.tensor(np.stack([src, dst]), dtype=torch.long),
             y=torch.tensor(labels if n == N else labels[:n], dtype=torch.long))
    for name, ix in (("train_mask", tr), ("val_mask", va), ("test_mask", te)):
        mk = torch.zeros(n, dtype=torch.bool); mk[ix] = True; setattr(g, name, mk)
    return g


class BrainTumorGAT(torch.nn.Module):
    def __init__(self, in_ch, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_ch, 512, heads=heads, dropout=0.4, concat=True); self.bn1 = torch.nn.BatchNorm1d(512 * heads)
        self.conv2 = GATConv(512 * heads, 256, heads=heads, dropout=0.4, concat=True); self.bn2 = torch.nn.BatchNorm1d(256 * heads)
        self.conv3 = GATConv(256 * heads, 128, heads=1, dropout=0.4, concat=False); self.bn3 = torch.nn.BatchNorm1d(128)
        self.classifier = torch.nn.Linear(128, 4); self.dropout = torch.nn.Dropout(0.4)
    def forward(self, data):
        x, ei = data.x, data.edge_index
        ei, _ = dropout_edge(ei, p=0.10, training=self.training)
        x = self.dropout(F.elu(self.bn1(self.conv1(x, ei))))
        x = self.dropout(F.elu(self.bn2(self.conv2(x, ei))))
        return self.classifier(F.elu(self.bn3(self.conv3(x, ei))))


class BrainTumorGCN(torch.nn.Module):
    """Control: same widths, plain graph convolution (every neighbor weighted by degree only, no attention)."""
    def __init__(self, in_ch, heads=4):
        super().__init__()
        self.conv1 = GCNConv(in_ch, 512 * heads); self.bn1 = torch.nn.BatchNorm1d(512 * heads)
        self.conv2 = GCNConv(512 * heads, 256 * heads); self.bn2 = torch.nn.BatchNorm1d(256 * heads)
        self.conv3 = GCNConv(256 * heads, 128); self.bn3 = torch.nn.BatchNorm1d(128)
        self.classifier = torch.nn.Linear(128, 4); self.dropout = torch.nn.Dropout(0.4)
    def forward(self, data):
        x, ei = data.x, data.edge_index
        ei, _ = dropout_edge(ei, p=0.10, training=self.training)
        x = self.dropout(F.elu(self.bn1(self.conv1(x, ei))))
        x = self.dropout(F.elu(self.bn2(self.conv2(x, ei))))
        return self.classifier(F.elu(self.bn3(self.conv3(x, ei))))


def _class_weights(y):
    c = np.bincount(y, minlength=4).astype(float)
    w = 1.0 / np.maximum(c, 1); w = w / w.sum() * 4
    return torch.tensor(w, dtype=torch.float, device=device)


def train_graph_model(graph, seed, budget, class_weights=True, arch="gat"):
    set_seed(seed)
    g = graph.to(device)
    model = (BrainTumorGAT if arch == "gat" else BrainTumorGCN)(g.x.shape[1]).to(device)
    w = _class_weights(g.y[g.train_mask].cpu().numpy()) if class_weights else None
    crit = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    epochs, patience = budget
    best, best_state, bad = float("inf"), None, 0
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        out = model(g); crit(out[g.train_mask], g.y[g.train_mask]).backward(); opt.step(); sch.step()
        model.eval()
        with torch.no_grad():
            out = model(g); pred = out.argmax(1)
            for nm, mk in (("train", g.train_mask), ("val", g.val_mask)):
                hist[f"{nm}_loss"].append(crit(out[mk], g.y[mk]).item())
                hist[f"{nm}_acc"].append((pred[mk] == g.y[mk]).float().mean().item())
        if hist["val_loss"][-1] < best:
            best, best_state, bad = hist["val_loss"][-1], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, hist


@torch.no_grad()
def predict_graph(model, graph):
    model.eval(); g = graph.to(device)
    P = F.softmax(model(g), 1).cpu().numpy()
    return P.argmax(1), P


class FeatureMLP(torch.nn.Module):
    def __init__(self, in_ch, heads=4, p=0.4):
        super().__init__()
        self.fc1 = torch.nn.Linear(in_ch, 512 * heads); self.bn1 = torch.nn.BatchNorm1d(512 * heads)
        self.fc2 = torch.nn.Linear(512 * heads, 256 * heads); self.bn2 = torch.nn.BatchNorm1d(256 * heads)
        self.fc3 = torch.nn.Linear(256 * heads, 128); self.bn3 = torch.nn.BatchNorm1d(128)
        self.classifier = torch.nn.Linear(128, 4); self.dropout = torch.nn.Dropout(p)
    def forward(self, x):
        x = self.dropout(F.elu(self.bn1(self.fc1(x))))
        x = self.dropout(F.elu(self.bn2(self.fc2(x))))
        return self.classifier(F.elu(self.bn3(self.fc3(x))))


def train_mlp(feats, tr, va, seed, budget):
    set_seed(seed)
    xt, xv = torch.tensor(feats[tr], device=device), torch.tensor(feats[va], device=device)
    yt, yv = torch.tensor(labels[tr], device=device), torch.tensor(labels[va], device=device)
    model = FeatureMLP(feats.shape[1]).to(device)
    crit = torch.nn.CrossEntropyLoss(weight=_class_weights(labels[tr]), label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    epochs, patience = budget
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad(); crit(model(xt), yt).backward(); opt.step(); sch.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(xv), yv).item()
        if vl < best:
            best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict_mlp(model, feats):
    model.eval()
    P = F.softmax(model(torch.tensor(feats, device=device)), 1).cpu().numpy()
    return P.argmax(1), P


def knn_vote(feats, ref_idx, query_idx, k):
    """Label of the k most similar labeled images (cosine), majority vote; ties go to the higher summed similarity.
    No training at all: if this matches the GAT, the graph is mostly looking things up."""
    Fn = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-12)
    R = torch.tensor(Fn[ref_idx], device=device); out = np.empty(len(query_idx), int)
    for s in range(0, len(query_idx), 1024):
        q = query_idx[s:s + 1024]
        S = torch.tensor(Fn[q], device=device) @ R.T
        same = torch.tensor(q, device=device)[:, None] == torch.tensor(ref_idx, device=device)[None, :]
        S = S.masked_fill(same, -2.0)
        sim, j = S.topk(k, dim=1)
        lab = torch.tensor(labels[ref_idx], device=device)[j]
        score = torch.zeros(len(q), 4, device=device)
        score.scatter_add_(1, lab, torch.ones_like(sim) + 1e-3 * sim)
        out[s:s + 1024] = score.argmax(1).cpu().numpy()
    full = np.full(N, -1); full[query_idx] = out
    return full

In [ ]:
# ===== 9. Metrics and tests =====
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def report(idx, pred_full, prob_full=None):
    y, p = labels[idx], pred_full[idx]
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=[0, 1, 2, 3], zero_division=0)
    k = int((y == p).sum()); present = sorted(set(y.tolist()))
    r = {"n": len(idx), "correct": k, "acc": k / len(idx), "wilson95": wilson(k, len(idx)),
         "kappa": float(cohen_kappa_score(y, p)), "macro_p": float(pr[present].mean()), "macro_r": float(rc[present].mean()),
         "macro_f1": float(f1_score(y, p, average="macro", labels=sorted(set(y)), zero_division=0)),
         "per_class": {c: {"p": float(pr[i]), "r": float(rc[i]), "f1": float(f1[i]), "n": int(sup[i])} for i, c in enumerate(CLASSES)},
         "cm": confusion_matrix(y, p, labels=[0, 1, 2, 3]).tolist()}
    if prob_full is not None:
        try:
            r["auc"] = {c: float(roc_auc_score((y == i).astype(int), prob_full[idx, i])) for i, c in enumerate(CLASSES) if 0 < (y == i).sum() < len(y)}
        except Exception:
            pass
    return r


def mcnemar(idx, pa, pb):
    y = labels[idx]; ca, cb = pa[idx] == y, pb[idx] == y
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    d = ca.astype(float) - cb.astype(float); se = d.std(ddof=1) / math.sqrt(len(d))
    return {"both_correct": int((ca & cb).sum()), "only_a": b, "only_b": c, "both_wrong": int((~ca & ~cb).sum()),
            "p_exact": float(binomtest(min(b, c), b + c, 0.5).pvalue) if b + c else 1.0,
            "diff_pp": 100 * d.mean(), "diff_ci95_pp": (100 * (d.mean() - 1.96 * se), 100 * (d.mean() + 1.96 * se))}


def acc_by_copy(idx, d, preds):
    out = {}
    for T in CFG["audit_T"]:
        f = d <= T
        out[int(T)] = {"n_with": int(f.sum()), "n_without": int((~f).sum())}
        for name, p in preds.items():
            c = p[idx] == labels[idx]
            out[int(T)][name] = {"with": float(c[f].mean()) if f.any() else None, "without": float(c[~f].mean()) if (~f).any() else None}
    return out


@torch.no_grad()
def attention_summary(model, graph, nodes, d_pair_T=None):
    """Per-layer attention over each node's incoming edges (self-loop included), for the given nodes.
    ratio = largest weight / (1 / number of incoming edges); 1.0 means perfectly equal weights."""
    model.eval(); g = graph.to(device)
    x, ei = g.x, g.edge_index
    h1, (e1, a1) = model.conv1(x, ei, return_attention_weights=True); x1 = F.elu(model.bn1(h1))
    h2, (e2, a2) = model.conv2(x1, ei, return_attention_weights=True); x2 = F.elu(model.bn2(h2))
    _, (e3, a3) = model.conv3(x2, ei, return_attention_weights=True)
    nodes_set = np.zeros(graph.num_nodes, bool); nodes_set[nodes] = True
    out = {}
    for nm, (e, a) in (("layer1", (e1, a1)), ("layer2", (e2, a2)), ("layer3", (e3, a3))):
        e = e.cpu().numpy(); a = a.mean(1).cpu().numpy() if a.dim() > 1 else a.cpu().numpy()
        keep_e = nodes_set[e[1]]
        df = pd.DataFrame({"dst": e[1][keep_e], "w": a[keep_e]})
        grp = df.groupby("dst").w
        deg = grp.size(); mx = grp.max()
        ent = grp.apply(lambda w: float(-(w * np.log(w + 1e-12)).sum()))
        out[nm] = {"mean_in_degree": float(deg.mean()), "mean_max_over_uniform": float((mx * deg).mean()),
                   "p90_max_over_uniform": float((mx * deg).quantile(0.9)),
                   "mean_norm_entropy": float((ent / np.log(deg.clip(lower=2))).mean()),
                   "share_nodes_max_within_10pct_of_uniform": float(((mx * deg) <= 1.10).mean())}
    # neighbors (graph edges, self excluded) of these nodes
    e = graph.edge_index.cpu().numpy(); m = nodes_set[e[1]] & (e[0] != e[1])
    s, t = e[0][m], e[1][m]
    same_label = labels[s] == labels[t]
    xor = lambda A, Bm: (A[s] != Bm[t]).sum(1)
    dmin = np.minimum.reduce([xor(B_O, B_O), xor(B_O, B_H), xor(B_O, B_V)])
    nb = pd.DataFrame({"t": t, "same": same_label, "copy": dmin <= CFG["near_T"], "test_src": graph.test_mask.cpu().numpy()[s]})
    agg = nb.groupby("t").agg(same=("same", "mean"), any_copy=("copy", "max"), test_src=("test_src", "mean"))
    out["neighbors"] = {"mean_share_same_label": float(agg.same.mean()), "share_all_neighbors_same_label": float((agg.same == 1).mean()),
                        "share_nodes_with_near_copy_neighbor": float(agg.any_copy.mean()),
                        "mean_share_neighbors_that_are_test": float(agg.test_src.mean())}
    return out, agg


def neighbor_case_study(model, graph, tr, va, te, pred, tag, n_per_class=1, top_k=4):
    """Saves thumbnails + metadata for one correctly classified test image per class and its top-4 neighbors (final layer)."""
    model.eval(); g = graph.to(device)
    with torch.no_grad():
        x, ei = g.x, g.edge_index
        x1 = F.elu(model.bn1(model.conv1(x, ei))); x2 = F.elu(model.bn2(model.conv2(x1, ei)))
        _, (e3, a3) = model.conv3(x2, ei, return_attention_weights=True)
    e3 = e3.cpu().numpy(); a3 = a3.mean(1).cpu().numpy() if a3.dim() > 1 else a3.cpu().numpy()
    split_of = np.full(N, "test", dtype=object); split_of[tr] = "train"; split_of[va] = "val"
    cases = []
    for ci, cls in enumerate(CLASSES):
        cand = [i for i in te if labels[i] == ci and pred[i] == ci][:n_per_class]
        for q in cand:
            m = (e3[1] == q) & (e3[0] != q)
            nb, w = e3[0][m], a3[m]; order = np.argsort(-w)[:top_k]
            items = [{"role": "query", "idx": int(q), "label": cls, "split": "test", "weight": None, "hamming": 0}]
            for j in order:
                n_ = int(nb[j])
                hd = int(min((B_O[q] != B_O[n_]).sum(), (B_H[q] != B_O[n_]).sum(), (B_V[q] != B_O[n_]).sum()))
                items.append({"role": "neighbor", "idx": n_, "label": CLASSES[labels[n_]], "split": split_of[n_],
                              "weight": float(w[j]), "hamming": hd})
            for it in items:
                fn = f"{tag}_{cls}_{it['role']}_{it['idx']}.png"
                Image.open(image_paths[it["idx"]]).convert("L").resize((224, 224)).save(os.path.join(THUMBS, fn))
                it["thumb"] = fn
            cases.append({"class": cls, "items": items})
    return cases

## Part A: main run on the random split and on the near-copy-free group split

In [ ]:
def full_run(kind, seed, bb_budget, gnn_budget, save_tag=None, keep=False):
    tr, va, te = make_split(kind, seed)
    info, d_te, _ = split_info(tr, va, te)
    bb = train_backbone(tr, va, seed, bb_budget)
    p_res, pr_res = predict_backbone(bb, te)
    feats = extract_features(bb)
    preds, probs, extra = {"resnet": p_res}, {"resnet": pr_res}, {}
    mlp = train_mlp(feats, tr, va, seed, gnn_budget)
    pm, pp = predict_mlp(mlp, feats); preds["mlp"], probs["mlp"] = pm, pp
    gT = build_graph(feats, tr, va, te, CFG["k"], "transductive")
    gat_t, hist_t = train_graph_model(gT, seed, gnn_budget)
    preds["gat_t"], probs["gat_t"] = predict_graph(gat_t, gT)
    gI = build_graph(feats, tr, va, te, CFG["k"], "inductive")
    gat_i, _ = train_graph_model(gI, seed, gnn_budget)
    preds["gat_i"], probs["gat_i"] = predict_graph(gat_i, gI)
    ref = np.concatenate([tr, va])
    preds["knn1"] = knn_vote(feats, ref, te, 1)
    preds["knn7"] = knn_vote(feats, ref, te, 7)
    res = {"split": info, "backbone_best_val_acc": bb["best_val_acc"], "backbone_epochs": len(bb["history"]),
           "backbone_fp16": bb["amp"],
           "models": {m: report(te, preds[m], probs.get(m)) for m in preds}}
    if save_tag:
        res["mcnemar"] = {f"{a}_vs_{b}": mcnemar(te, preds[a], preds[b]) for a, b in
                          (("gat_t", "resnet"), ("gat_t", "mlp"), ("mlp", "resnet"), ("gat_i", "mlp"), ("gat_t", "knn7"))}
        res["acc_by_near_copy"] = acc_by_copy(te, d_te, preds)
        res["gat_t_history"] = hist_t
        res["gat_t_val_acc"] = float((preds["gat_t"][va] == labels[va]).mean())
        res["backbone_history"] = bb["history"]
        att, agg = attention_summary(gat_t, gT, te)
        res["attention_gat_t"] = att
        has_copy_nb = np.zeros(N, bool); has_copy_nb[agg.index.values[agg.any_copy.values.astype(bool)]] = True
        c = preds["gat_t"][te] == labels[te]
        res["gat_t_acc_by_copy_neighbor"] = {"with": float(c[has_copy_nb[te]].mean()) if has_copy_nb[te].any() else None,
                                             "without": float(c[~has_copy_nb[te]].mean()) if (~has_copy_nb[te]).any() else None,
                                             "n_with": int(has_copy_nb[te].sum())}
        res["neighbor_cases"] = neighbor_case_study(gat_t, gT, tr, va, te, preds["gat_t"], save_tag)
        res["params"] = {"resnet": int(sum(p.numel() for p in bb["model"].parameters())),
                         "gat": int(sum(p.numel() for p in gat_t.parameters())),
                         "mlp": int(sum(p.numel() for p in mlp.parameters()))}
        df = pd.DataFrame({"idx": te, "label": labels[te], "source": image_source[te], "near_copy_dist": d_te,
                           **{m: preds[m][te] for m in preds}})
        df.to_csv(os.path.join(OUT, f"preds_{save_tag}.csv"), index=False)
    log(f"[{kind} seed {seed}] " + " | ".join(f"{m} {res['models'][m]['acc']:.4f}" for m in preds) +
        f" | test near-copy share (<= {CFG['near_T']} bits) {info['test_near_copy_share'][CFG['near_T']]:.3f}")
    state = dict(tr=tr, va=va, te=te, bb=bb, feats=feats, gT=gT, gat_t=gat_t, preds=preds) if keep else None
    if not keep:
        del bb, feats, gat_t, gat_i, mlp
        torch.cuda.empty_cache()
    return res, state


MAIN = {}


def stage_A():
    for kind in split_kinds():
        res, st = full_run(kind, CFG["main_seed"], CFG["bb_main"], CFG["gnn_main"], save_tag=f"A_{kind}", keep=kind == "random")
        RESULTS.setdefault("A", {})[kind] = res
        if st:
            MAIN[kind] = st
        save_results()


if CFG["run"]["A"]:
    run_stage("A_main_random_and_group", stage_A)

In [ ]:
# ===== Part A extras: efficiency, Grad-CAM figure (random split backbone) =====
def stage_A_extras():
    st = MAIN["random"]; m = st["bb"]["model"]; m.eval()
    x = eval_tf(enhance_mri(Image.open(image_paths[st["te"][0]]).convert("RGB"))).unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(3):
            m(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20):
            m(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    RESULTS["A"]["random"]["resnet_ms_per_image"] = (time.time() - t0) / 20 * 1000
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
    except Exception as e:
        log("Grad-CAM skipped:", e); return
    cam = GradCAM(model=m, target_layers=[m.layer4[-1]])
    fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
    meta = []
    for ci, cls in enumerate(CLASSES):
        for j, gi in enumerate([i for i in st["te"] if labels[i] == ci][:2]):
            img = enhance_mri(Image.open(image_paths[gi]).convert("RGB"))
            gc = cam(input_tensor=eval_tf(img).unsqueeze(0).to(device), targets=None)[0]
            vis = show_cam_on_image(np.array(img.resize((224, 224))).astype(np.float32) / 255.0, gc, use_rgb=True)
            Image.fromarray(vis).save(os.path.join(THUMBS, f"gradcam_{cls}_{j}_{gi}.png"))
            axes[j, ci].imshow(vis); axes[j, ci].axis("off")
            if j == 0:
                axes[j, ci].set_title(cls)
            meta.append({"class": cls, "idx": int(gi), "pred": CLASSES[int(st["preds"]["resnet"][gi])]})
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "gradcam_random_split.png"), dpi=150); plt.close()
    RESULTS["A"]["random"]["gradcam_cases"] = meta


if CFG["run"]["A"] and "random" in MAIN:
    run_stage("A_extras", stage_A_extras)

## Part B: ablation on the random split (single seed, short GAT budget, like the paper) plus a GCN control

In [ ]:
def stage_B():
    st = MAIN["random"]; tr, va, te, feats = st["tr"], st["va"], st["te"], st["feats"]
    rows = []
    def add(name, graph, **kw):
        m, _ = train_graph_model(graph, CFG["main_seed"], CFG["gnn_short"], **kw)
        p, _ = predict_graph(m, graph)
        rows.append({"config": name, "acc": float((p[te] == labels[te]).mean()), "correct": int((p[te] == labels[te]).sum())})
        log("ablation", rows[-1]); del m
    for mode in ("transductive", "inductive"):
        for k in (4, 8, 16):
            add(f"{mode} k={k}", build_graph(feats, tr, va, te, k, mode))
    add("class weights off (transductive k=8)", build_graph(feats, tr, va, te, 8, "transductive"), class_weights=False)
    add("GCN instead of GAT (transductive k=8)", build_graph(feats, tr, va, te, 8, "transductive"), arch="gcn")
    add("GCN instead of GAT (inductive k=8)", build_graph(feats, tr, va, te, 8, "inductive"), arch="gcn")
    f_single = extract_features(st["bb"], multiscale=False)
    add("final-layer features only (2048-d)", build_graph(f_single, tr, va, te, 8, "transductive"))
    bb0 = train_backbone(tr, va, CFG["main_seed"], CFG["bb_short"], use_clahe=False, amp=False)
    f0 = extract_features(bb0)
    add("CLAHE off (backbone retrained in fp32, 8-epoch budget)", build_graph(f0, tr, va, te, 8, "transductive"))
    p0, _ = predict_backbone(bb0, te)
    rows.append({"config": "CLAHE off: ResNet50 head", "acc": float((p0[te] == labels[te]).mean()), "correct": int((p0[te] == labels[te]).sum())})
    RESULTS["B_ablation"] = rows
    pd.DataFrame(rows).to_csv(os.path.join(OUT, "B_ablation.csv"), index=False)
    del bb0; torch.cuda.empty_cache()


if CFG["run"]["B"] and "random" in MAIN:
    run_stage("B_ablation", stage_B)

# free the big Part A objects we no longer need (keep nothing on the GPU)
for st in MAIN.values():
    if st:
        st["bb"]["model"].cpu(); st["gat_t"].cpu()
torch.cuda.empty_cache()

## Part C: five seeds on both splits, every model (the seed changes the split and the weights)

In [ ]:
def stage_C():
    path = os.path.join(OUT, "C_seeds.csv"); rows = []
    for seed in CFG["seeds"]:
        for kind in split_kinds():
            if not time_left_ok(margin_min=45):
                log("deadline near: stopping Part C early"); break
            res, _ = full_run(kind, seed, CFG["bb_short"], CFG["gnn_short"])
            for m, r in res["models"].items():
                rows.append({"seed": seed, "split": kind, "model": m, "acc": r["acc"], "macro_f1": r["macro_f1"],
                             "n_test": r["n"], "test_near_copy_share": res["split"]["test_near_copy_share"][CFG["near_T"]]})
            pd.DataFrame(rows).to_csv(path, index=False)
    df = pd.DataFrame(rows)
    summ = df.groupby(["split", "model"]).acc.agg(["mean", "std", "count"]).reset_index()
    RESULTS["C_seeds"] = {"rows": rows, "summary": summ.to_dict("records")}
    tests = {}
    piv = df.pivot_table(index=["split", "seed"], columns="model", values="acc")
    for kind in split_kinds():
        if kind not in piv.index.get_level_values(0):
            continue
        sub = piv.loc[kind]
        for a, b in (("gat_t", "mlp"), ("gat_t", "resnet"), ("mlp", "resnet"), ("gat_i", "mlp"), ("gat_t", "knn7")):
            if len(sub) >= 2:
                d = sub[a] - sub[b]
                tests[f"{kind}:{a}-{b}"] = {"mean_pp": 100 * d.mean(), "wins": int((d > 0).sum()), "n": int(len(d)),
                                            "p_paired_t": float(ttest_rel(sub[a], sub[b]).pvalue) if d.std() > 0 else 1.0}
    have = set(piv.index.get_level_values(0))
    for a_k, b_k in (("random", "group"), ("random", "patient"), ("group", "patient")):
        if not {a_k, b_k} <= have:
            continue
        for m in piv.columns:
            d = (piv.loc[a_k][m] - piv.loc[b_k][m]).dropna()
            if len(d) >= 2:
                tests[f"{a_k}-{b_k}:{m}"] = {"mean_pp": 100 * d.mean(), "wins": int((d > 0).sum()), "n": int(len(d)),
                                             "p_paired_t": float(ttest_rel(piv.loc[a_k][m].loc[d.index], piv.loc[b_k][m].loc[d.index]).pvalue) if d.std() > 0 else 1.0}
    RESULTS["C_seeds"]["paired_tests"] = tests
    log("Part C summary:\n" + summ.to_string(index=False))


if CFG["run"]["C"]:
    run_stage("C_seeds", stage_C)

## Part D: leave-one-source-out, 5 seeds, per-image predictions (so each held-out source can be rescored without its near-copies)

In [ ]:
def make_inductive_graphs(feats, tr, va, held, k=8):
    """Strictly inductive: training graph = training-source nodes only (k real neighbors, self excluded);
    each held-out image receives edges from its k nearest training nodes and sends none."""
    allowed = np.concatenate([tr, va]); A, H = len(allowed), len(held)
    nn_ = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(feats[allowed])
    _, ia = nn_.kneighbors(feats[allowed])
    ia = np.stack([row[row != i][:k] for i, row in enumerate(ia)])
    _, ih = NearestNeighbors(n_neighbors=k, metric="cosine").fit(feats[allowed]).kneighbors(feats[held])
    e_tr = np.stack([ia.reshape(-1), np.repeat(np.arange(A), k)])
    e_full = np.concatenate([e_tr, np.stack([ih.reshape(-1), np.repeat(np.arange(H), k) + A])], 1)
    assert (e_full[0] < A).all()
    def mk(x, y, e, n_train, n_val, test_from):
        g = Data(x=torch.tensor(x, dtype=torch.float), edge_index=torch.tensor(e, dtype=torch.long), y=torch.tensor(y, dtype=torch.long))
        n = len(x)
        g.train_mask = torch.zeros(n, dtype=torch.bool); g.train_mask[:n_train] = True
        g.val_mask = torch.zeros(n, dtype=torch.bool); g.val_mask[n_train:n_train + n_val] = True
        g.test_mask = torch.zeros(n, dtype=torch.bool)
        if test_from is not None:
            g.test_mask[test_from:] = True
        return g
    g_tr = mk(feats[allowed], labels[allowed], e_tr, len(tr), len(va), None)
    g_full = mk(np.concatenate([feats[allowed], feats[held]]), np.concatenate([labels[allowed], labels[held]]), e_full, len(tr), len(va), A)
    return g_tr, g_full


def heldout_run(train_pool, test_idx, seed, tag_extra):
    tr, va = train_test_split(np.asarray(train_pool), test_size=0.2, random_state=seed, stratify=labels[train_pool])
    tr, va = np.sort(tr), np.sort(va)
    bb = train_backbone(tr, va, seed, CFG["bb_short"])
    preds = {}
    preds["resnet"], _ = predict_backbone(bb, test_idx)
    feats = extract_features(bb); del bb; torch.cuda.empty_cache()
    mlp = train_mlp(feats, tr, va, seed, CFG["gnn_short"]); preds["mlp"], _ = predict_mlp(mlp, feats)
    g_tr, g_full = make_inductive_graphs(feats, tr, va, test_idx, CFG["k"])
    gat, _ = train_graph_model(g_tr, seed, CFG["gnn_short"])
    pg, _ = predict_graph(gat, g_full)
    full = np.full(N, -1); full[test_idx] = pg[len(tr) + len(va):]; preds["gat_i"] = full
    ref = np.concatenate([tr, va])
    preds["knn1"] = knn_vote(feats, ref, np.asarray(test_idx), 1)
    preds["knn7"] = knn_vote(feats, ref, np.asarray(test_idx), 7)
    del feats, mlp, gat; torch.cuda.empty_cache()
    return preds, len(tr) + len(va)


def stage_D():
    sources = sorted(set(image_source)); rows, per_img = [], []
    cross = {}
    for s in sources:
        held = np.where(image_source == s)[0]; pool = np.where(image_source != s)[0]
        d, _, _ = nearest_copy(held, pool); cross[s] = d
        RESULTS.setdefault("D_cross_source_near_copy", {})[s] = {int(T): float((d <= T).mean()) for T in CFG["audit_T"]}
    for s in sources:
        held = np.where(image_source == s)[0]; pool = np.where(image_source != s)[0]
        for seed in CFG["seeds"]:
            if not time_left_ok(margin_min=40):
                log("deadline near: stopping Part D early"); break
            preds, n_train = heldout_run(pool, held, seed, s)
            clean = cross[s] > CFG["near_T"]
            for m, p in preds.items():
                c = p[held] == labels[held]
                rows.append({"held_out": s, "seed": seed, "model": m, "n_test": len(held), "n_train_pool": n_train,
                             "acc": float(c.mean()), "macro_f1": float(f1_score(labels[held], p[held], average="macro", labels=sorted(set(labels[held])), zero_division=0)),
                             "n_clean": int(clean.sum()), "acc_clean": float(c[clean].mean()) if clean.any() else None,
                             "acc_with_copy": float(c[~clean].mean()) if (~clean).any() else None})
            per_img.append(pd.DataFrame({"held_out": s, "seed": seed, "idx": held, "label": labels[held], "cross_dist": cross[s],
                                         **{m: p[held] for m, p in preds.items()}}))
            pd.DataFrame(rows).to_csv(os.path.join(OUT, "D_loso.csv"), index=False)
            pd.concat(per_img).to_csv(os.path.join(OUT, "D_loso_per_image.csv"), index=False)
            log(f"[LOSO {s} seed {seed}] (all / no cross-source copy) " + " | ".join(
                f"{r['model']} {r['acc']:.4f}/" + (f"{r['acc_clean']:.4f}" if r['acc_clean'] is not None else "na") for r in rows[-len(preds):]))
    df = pd.DataFrame(rows)
    RESULTS["D_loso"] = {"rows": rows,
                         "summary": df.groupby(["held_out", "model"]).agg(acc=("acc", "mean"), acc_sd=("acc", "std"),
                                                                           acc_clean=("acc_clean", "mean"), f1=("macro_f1", "mean")).reset_index().to_dict("records")}


if CFG["run"]["D"]:
    run_stage("D_leave_one_source_out", stage_D)

## Part E: is the MRI.zip-held-out drop just a small, lopsided training set?
The MRI.zip fold trains on BRISC + PMRAM only (about 2,000 images, mostly no-tumor, few pituitary).
Here the same number of images per class is drawn from all three sources (group split, so no near-copies
of the test set), and the model is tested in-distribution. If accuracy stays high, size and class mix do not explain the drop.

In [ ]:
def stage_E():
    held_src = "ds1"
    pool_loso = np.where(image_source != held_src)[0]
    target = np.bincount(labels[pool_loso], minlength=4)
    RESULTS["E_size_control"] = {"target_per_class": {c: int(target[i]) for i, c in enumerate(CLASSES)}, "rows": []}
    rows = []
    for seed in CFG["seeds"]:
        if not time_left_ok(margin_min=30):
            log("deadline near: stopping Part E early"); break
        tr, va, te = make_split("patient" if PATIENT_OK else "group", seed)
        ref = np.concatenate([tr, va]); rng = np.random.RandomState(seed); pick = []
        for ci in range(4):
            cand = ref[labels[ref] == ci]
            pick.append(rng.choice(cand, size=min(int(target[ci]), len(cand)), replace=False))
        pick = np.sort(np.concatenate(pick))
        preds, n_train = heldout_run(pick, te, seed, "E")
        for m, p in preds.items():
            c = p[te] == labels[te]; ds1 = image_source[te] == held_src
            rows.append({"seed": seed, "model": m, "n_train_pool": n_train, "acc_group_test": float(c.mean()),
                         "acc_group_test_ds1_only": float(c[ds1].mean()) if ds1.any() else None,
                         "macro_f1": float(f1_score(labels[te], p[te], average="macro", zero_division=0))})
        pd.DataFrame(rows).to_csv(os.path.join(OUT, "E_size_control.csv"), index=False)
        log(f"[size control seed {seed}] " + " | ".join(f"{r['model']} {r['acc_group_test']:.4f}" for r in rows[-len(preds):]))
    RESULTS["E_size_control"]["rows"] = rows


if CFG["run"]["E"]:
    run_stage("E_size_control", stage_E)

In [ ]:
# ===== Export: one zip with everything =====
import shutil
RESULTS["total_minutes"] = round((time.time() - T_START) / 60, 1)
save_results()
zip_base = os.path.join(WORK, "msgat_v5_results")
shutil.make_archive(zip_base, "zip", OUT)
log("Saved", zip_base + ".zip", "| errors:", {k: v.splitlines()[0] for k, v in RESULTS["errors"].items()} or "none")
print("\nDone. Download msgat_v5_results.zip from the Output tab and send it to Claude.")